In [0]:
from pyspark.sql.functions import explode, from_json

# ADLS source path
source_dir = "abfss://source-adls@adlsmewalth.dfs.core.windows.net/"
source_file = "ecommerce_financials*.json"

# Schema tracking location (required for Auto Loader)
schema_location = "abfss://source-adls@adlsmewalth.dfs.core.windows.net/checkpoints/schema/ecommerce_v2"

# Read JSON files using Auto Loader
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("pathGlobFilter", source_file)
    .load(source_dir)
)

# -----------------------------
# Metadata DataFrame
# -----------------------------
metadata_df = df.select(
    "metadata.generated_at",
    "metadata.label",
    "metadata.total_records"
)

# -----------------------------
# Profit Loss DataFrame
# -----------------------------
profit_loss_df = (
    df.select(explode("profit_loss").alias("PL"))
      .select(
          "PL.cost_price",
          "PL.gst_rate",
          "PL.net_profit",
          "PL.order_id",
          "PL.payout_status",
          "PL.platform_fee",
          "PL.product_id",
          "PL.selling_price",
          "PL.tax",
          "PL.vendor_id",
          "PL.commission_amount"
      )
)

# -----------------------------
# Commission DataFrame
# -----------------------------
commission_df = (
    df.select(explode("commissions").alias("commission"))
      .select(
          "commission.commission_amount",
          "commission.commission_percentage",
          "commission.order_date",
          "commission.order_id",
          "commission.product_id",
          "commission.vendor_id"
      )
)

# -----------------------------
# Vendor Payouts DataFrame
# -----------------------------
vendor_payouts_df = (
    df.select(explode("vendor_payouts").alias("VP"))
      .select("VP.*")
)

# =========================================================
# Write Streams to Delta Tables
# =========================================================

# Profit Loss Table
(
    profit_loss_df.writeStream
    .trigger(availableNow=True)
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://source-adls@adlsmewalth.dfs.core.windows.net/checkpoints/profit_loss"
    )
    .table("ekart_dt.bronze.profit_loss")
)

# Commission Table
(
    commission_df.writeStream
    .trigger(availableNow=True)
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://source-adls@adlsmewalth.dfs.core.windows.net/checkpoints/commission"
    )
    .table("ekart_dt.bronze.commission")
)

# Vendor Payout Table
(
    vendor_payouts_df.writeStream
    .trigger(availableNow=True)
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://source-adls@adlsmewalth.dfs.core.windows.net/checkpoints/vendor_payouts"
    )
    .table("ekart_dt.bronze.vendor_payouts")
)

# Metadata Table
(
    metadata_df.writeStream
    .trigger(availableNow=True)
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://source-adls@adlsmewalth.dfs.core.windows.net/checkpoints/metadata"
    )
    .table("ekart_dt.bronze.metadata")
)